# Bounded WBCE on CheXpert

Select a CUDA GPU and set the paths below. The training cell starts a new experiment; saved models are then used for evaluation. See the README for the protocol.


In [ ]:
from pathlib import Path
import subprocess, sys
REPO = Path('/content/chexpert-bounded-wbce')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/GPedreira/chexpert-bounded-wbce.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements.txt')],check=True)
print('Commit:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = Path('/content/chexpert')
OUTPUT = Path('/content/drive/MyDrive/WVC/experiment')
PHASE = 'all'  # Use '2b' for only the nine-model comparison.
RESUME = False


## Training
This cell starts GPU training. Use an empty output directory for a new experiment, or set RESUME=True to continue the same experiment.


In [ ]:
command = [sys.executable,str(REPO/'wvc.py'),'train','--data-dir',str(DATA),
           '--output-root',str(OUTPUT),'--phase',PHASE]
if RESUME:
    command.append('--resume')
subprocess.run(command,check=True)


In [ ]:
EVALUATION_ROOT = OUTPUT/'evaluation'
previous = set(EVALUATION_ROOT.iterdir()) if EVALUATION_ROOT.exists() else set()
subprocess.run([sys.executable,str(REPO/'wvc.py'),'evaluate','--data-dir',str(DATA),
                '--experiment-root',str(OUTPUT),'--output-root',str(EVALUATION_ROOT)],check=True)
created = [p for p in EVALUATION_ROOT.iterdir() if p not in previous and (p/'manifest.json').exists()]
if len(created) != 1:
    raise RuntimeError('Expected one completed evaluation directory.')
EVALUATION = created[0]
subprocess.run([sys.executable,str(REPO/'wvc.py'),'figures','--evaluation-dir',str(EVALUATION)],check=True)
from IPython.display import Image, display
display(Image(filename=str(EVALUATION/'figure_tail_fp32.png')))
